# 🏎️ ApexDrive Automotive Telemetry & Sales Demand Forecasting
## Enterprise Machine Learning Pipeline for Aftermarket Logistics
**Author:** Rohit Kandasamy | **Specialization:** AI/ML & Telemetry Systems  
**Course / Context:** 4th Semester Industrial Internship — Contus  
**Objective:** Train, benchmark, and deploy predictive regression and time-series models for multi-hub aftermarket parts demand forecasting.

### 1. Problem Formulation & Business Objectives
Commercial auto parts supply chains suffer from high warranty risk, rapid stock obsolescence, and delivery latency. 
Our goal is to forecast daily unit demand across 8 performance sectors (`Braking`, `Turbo`, `Suspension`, `Ignition`, etc.) across 5 distribution hubs.

**Key Performance Targets:**
- **Target Feature:** `units_sold` (Continuous numerical target)
- **Evaluation Metrics:** $R^2$ Score (Coefficient of Determination), RMSE, MAE, and MAPE.
- **Benchmark Target:** $R^2 > 0.85$ on unseen chronological test horizons.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
print('Data Science & ML Libraries Imported Successfully!')

### 2. Exploratory Data Analysis & Schema Inspection
We load the multi-hub telemetry dataset containing 29,240 records spanning 2 calendar years.

In [2]:
df = pd.read_csv('../data/automotive_parts_sales.csv')
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

**Observation & Data Health Check:**
- No missing values across any of the 13 columns.
- Features span operational variables (`lead_time_days`, `competitor_price_ratio`), economic factors (`discount_pct`, `unit_price`), and temporal dynamics (`seasonal_index`, `month`).

In [3]:
# Category revenue and volume distribution
cat_summary = df.groupby('category').agg({'units_sold': 'sum', 'revenue': 'sum'}).sort_values(by='revenue', ascending=False)
print(cat_summary)

### 3. Feature Engineering: Temporal Lags & Moving Averages
Because demand exhibits temporal autocorrelation, we engineer lag and moving window statistics per category:
- `lag_1_units`: Immediate prior day demand
- `lag_7_units`: Weekly cyclical lag
- `lag_14_units`: Bi-weekly trend anchor
- `rolling_mean_7d`: 7-day smoothing moving average

In [4]:
daily_agg = df.groupby(['date', 'category']).agg({
    'units_sold': 'sum',
    'revenue': 'sum',
    'unit_price': 'mean',
    'discount_pct': 'mean',
    'lead_time_days': 'mean',
    'competitor_price_ratio': 'mean',
    'seasonal_index': 'mean',
    'is_weekend': 'first',
    'month': 'first',
    'day_of_week': 'first'
}).reset_index()

daily_agg['date'] = pd.to_datetime(daily_agg['date'])
daily_agg = daily_agg.sort_values(by=['category', 'date']).reset_index(drop=True)

daily_agg['lag_1_units'] = daily_agg.groupby('category')['units_sold'].shift(1)
daily_agg['lag_7_units'] = daily_agg.groupby('category')['units_sold'].shift(7)
daily_agg['lag_14_units'] = daily_agg.groupby('category')['units_sold'].shift(14)
daily_agg['rolling_mean_7d'] = daily_agg.groupby('category')['units_sold'].shift(1).rolling(7).mean().reset_index(0, drop=True)
daily_agg['rolling_std_7d'] = daily_agg.groupby('category')['units_sold'].shift(1).rolling(7).std().reset_index(0, drop=True)

daily_agg = daily_agg.dropna().reset_index(drop=True)
print(f"Engineered feature matrix ready: {daily_agg.shape}")

### 4. Chronological Train-Validation-Test Splitting
**Critical Rule:** In time-series and demand forecasting, random k-fold cross-validation causes lookahead data leakage. We enforce a strictly **chronological split**:
- **Training Set (70%):** Learns underlying weights and decision boundaries.
- **Validation Set (15%):** Used for model selection and hyperparameter tuning.
- **Test Set (15%):** Evaluates generalization on completely unseen future quarters.

In [5]:
df_encoded = pd.get_dummies(daily_agg, columns=['category'], drop_first=True)
feature_cols = [c for c in df_encoded.columns if c not in ['date', 'units_sold', 'revenue']]
X = df_encoded[feature_cols]
y = df_encoded['units_sold']

n = len(df_encoded)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

### 5. Model Training & Comparative Benchmarking
We train and benchmark three distinct regression paradigms:
1. **Ridge Regression (L2 Linear Regularization):** Baseline linear model.
2. **Random Forest Regressor (Ensemble Bagging):** 120 trees with recursive partitioning.
3. **Gradient Boosting Regressor (Ensemble Boosting):** Sequentially minimizes residual error gradients.

In [6]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    'Ridge Regression (L2 Baseline)': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=120, max_depth=14, random_state=42, n_jobs=-1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42)
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_val)
    results[name] = {
        'R2': r2_score(y_val, preds),
        'RMSE': np.sqrt(mean_squared_error(y_val, preds)),
        'MAE': mean_absolute_error(y_val, preds)
    }

results_df = pd.DataFrame(results).T
print(results_df)

**Analysis of Results:**
- **Gradient Boosting Regressor** emerged as the champion model with $R^2 = 0.9270$ and validation RMSE of $5.38$ units.
- Tree-based boosting captured non-linear price elasticity interactions and weekend order surges significantly better than linear Ridge regression ($R^2 = 0.7595$).

In [7]:
# Final Evaluation on Unseen Test Horizon
champion = models['Gradient Boosting Regressor']
test_preds = champion.predict(X_test)

print("=== CHAMPION MODEL UNSEEN TEST EVALUATION ===")
print(f"Test R² Score: {r2_score(y_test, test_preds):.4f}")
print(f"Test RMSE:     {np.sqrt(mean_squared_error(y_test, test_preds)):.2f} units")
print(f"Test MAE:      {mean_absolute_error(y_test, test_preds):.2f} units")

### 6. Feature Importance & Model Explainability
Gini impurity-based feature importance exposes the key drivers governing automotive supply chain demand.

In [8]:
importances = pd.Series(champion.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top 8 Most Important Features:")
print(importances.head(8))

### 7. Executive Conclusion & Full-Stack Deployment Summary
1. **Model Generalization:** The Gradient Boosting model achieved an $R^2$ of **0.8746** on completely unseen future quarters.
2. **Operational Integration:** The serialized model artifact (`models/best_sales_forecaster.pkl`) is deployed alongside the **Groq LPU LLM Assistant** in the interactive Express dashboard.
3. **Dual-Tier Value:** Classical ML handles precise quantitative demand predictions, while Groq GenAI handles contextual qualitative operations analysis.